In [1]:
import numpy as np
import pandas as pd
from yaml import safe_load
import os
import pickle
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

print('All imports successful!')

All imports successful!


In [2]:
filenames = [
    os.path.join('data', f)
    for f in os.listdir('data')
    if f.endswith('.yaml')
]
print(f'Total files: {len(filenames)}')

Total files: 4010


In [3]:
frames = []
counter = 1

for file in tqdm(filenames):
    with open(file, 'r') as f:
        data = safe_load(f)
    if isinstance(data, dict):
        data = [data]
    elif not isinstance(data, list):
        print(f'Skipping {file}: unexpected type {type(data)}')
        counter += 1
        continue
    df = pd.json_normalize(data)
    df['match_id'] = counter
    frames.append(df)
    counter += 1

def chunked_concat(frames, chunk_size=500):
    chunks = []
    for i in range(0, len(frames), chunk_size):
        chunk = pd.concat(frames[i:i+chunk_size], ignore_index=True)
        chunks.append(chunk)
        print(f'Chunk {i//chunk_size + 1} done')
    return pd.concat(chunks, ignore_index=True)

final_df = chunked_concat(frames)
print('Done! Completed successfully.')
print(final_df.shape)

100%|██████████| 4010/4010 [17:28<00:00,  3.83it/s]


Chunk 1 done
Chunk 2 done
Chunk 3 done
Chunk 4 done
Chunk 5 done
Chunk 6 done
Chunk 7 done
Chunk 8 done
Chunk 9 done
Done! Completed successfully.
(4010, 4412)


In [4]:
final_df.drop(columns=[
    'meta.data_version', 'meta.created', 'meta.revision',
    'info.outcome.bowl_out', 'info.bowl_out',
    'info.supersubs.South Africa', 'info.supersubs.New Zealand',
    'info.outcome.eliminator', 'info.outcome.result',
    'info.outcome.method', 'info.neutral_venue',
    'info.match_type_number', 'info.outcome.by.runs',
    'info.outcome.by.wickets'
], inplace=True, errors='ignore')

print(final_df['info.gender'].value_counts())
print(final_df['info.match_type'].value_counts())
print(final_df['info.overs'].value_counts())

info.gender
female    4010
Name: count, dtype: int64
info.match_type
T20     3029
ODI      597
ODM      284
IT20      77
Test      23
Name: count, dtype: int64
info.overs
20.0    3099
50.0     888
Name: count, dtype: int64


In [5]:
final_df = final_df[final_df['info.gender'] == 'female']
final_df = final_df[final_df['info.match_type'] == 'T20']
final_df = final_df[final_df['info.overs'] == 20]
final_df.drop(columns=['info.gender', 'info.match_type', 'info.overs'], inplace=True)

print(f'Matches remaining: {len(final_df)}')
final_df.head()

Matches remaining: 3022


,innings,info.balls_per_over,info.competition,info.dates,info.outcome.winner,info.player_of_match,info.players.Melbourne Stars,info.players.Melbourne Renegades,info.registry.people.A King,info.registry.people.AK Wilds,...,info.registry.people.Shai Carrington,info.registry.people.J Brown,info.registry.people.S Campbelle,info.registry.people.A Freeman,info.registry.people.AM Pereira,info.registry.people.M Clarke,info.registry.people.K Ferron,info.registry.people.S Bellot,info.registry.people.K Persaud,info.registry.people.CN Whyte
0,"[{'1st innings': {'team': 'Melbourne Stars', '...",6,Women's Big Bash League,[2017-01-01],Melbourne Renegades,[RH Priest],"[MM Lanning, EJ Inglis, JE Cameron, NR Sciver,...","[RH Priest, S Molineux, DN Wyatt, KL Britt, GM...",83558266,de412f40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"[{'1st innings': {'team': 'Sydney Sixers', 'de...",6,Women's Big Bash League,[2017-01-02],Sydney Sixers,[AJ Healy],NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[{'1st innings': {'team': 'Sydney Thunder', 'd...",6,Women's Big Bash League,[2017-01-02],Sydney Thunder,[SL Bates],NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[{'1st innings': {'team': 'Adelaide Strikers',...",6,Women's Big Bash League,[2017-01-03],Sydney Sixers,[EA Perry],NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"[{'1st innings': {'team': 'Perth Scorchers', '...",6,Women's Big Bash League,[2017-01-04],Melbourne Renegades,[GM Harris],NaN,"[RH Priest, S Molineux, DN Wyatt, M Brown, KL ...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
pickle.dump(final_df, open('dataset_level1_female.pkl', 'wb'))
print('Saved dataset_level1_female.pkl')

Saved dataset_level1_female.pkl


In [7]:
matches = pickle.load(open('dataset_level1_female.pkl', 'rb'))

count = 1
delivery_frames = []

for index, row in tqdm(matches.iterrows(), total=matches.shape[0]):
    try:
        innings_data = row['innings'][0]['1st innings']['deliveries']
    except (KeyError, TypeError, IndexError):
        count += 1
        continue

    ball_of_match, batsman, bowler, runs = [], [], [], []
    player_dismissed, teams, batting_team = [], [], []
    match_id, city, venue = [], [], []

    for ball in innings_data:
        for key in ball.keys():
            match_id.append(count)
            batting_team.append(row['innings'][0]['1st innings']['team'])
            teams.append(row['info.teams'])
            ball_of_match.append(key)
            batsman.append(ball[key]['batsman'])
            bowler.append(ball[key]['bowler'])
            runs.append(ball[key]['runs']['total'])
            city.append(row.get('info.city', None))
            venue.append(row.get('info.venue', None))
            try:
                player_dismissed.append(ball[key]['wicket']['player_out'])
            except:
                player_dismissed.append('0')

    loop_df = pd.DataFrame({
        'match_id': match_id, 'teams': teams, 'batting_team': batting_team,
        'ball': ball_of_match, 'batsman': batsman, 'bowler': bowler,
        'runs': runs, 'player_dismissed': player_dismissed,
        'city': city, 'venue': venue
    })
    delivery_frames.append(loop_df)
    count += 1

delivery_df = pd.concat(delivery_frames, ignore_index=True)
print(f'Total deliveries: {len(delivery_df)}')
delivery_df.head()

100%|██████████| 3022/3022 [00:09<00:00, 314.00it/s]


Total deliveries: 368238


,match_id,teams,batting_team,ball,batsman,bowler,runs,player_dismissed,city,venue
0,1,"[Melbourne Stars, Melbourne Renegades]",Melbourne Stars,0.1,MM Lanning,LMM Tahuhu,0,0,NaN,Melbourne Cricket Ground
1,1,"[Melbourne Stars, Melbourne Renegades]",Melbourne Stars,0.2,MM Lanning,LMM Tahuhu,1,0,NaN,Melbourne Cricket Ground
2,1,"[Melbourne Stars, Melbourne Renegades]",Melbourne Stars,0.3,EJ Inglis,LMM Tahuhu,0,0,NaN,Melbourne Cricket Ground
3,1,"[Melbourne Stars, Melbourne Renegades]",Melbourne Stars,0.4,EJ Inglis,LMM Tahuhu,0,0,NaN,Melbourne Cricket Ground
4,1,"[Melbourne Stars, Melbourne Renegades]",Melbourne Stars,0.5,EJ Inglis,LMM Tahuhu,0,0,NaN,Melbourne Cricket Ground


In [8]:
def bowl(row):
    for team in row['teams']:
        if team != row['batting_team']:
            return team

delivery_df['bowling_team'] = delivery_df.apply(bowl, axis=1)
delivery_df.drop(columns=['teams'], inplace=True)

# ICC Women's T20 World Cup 2026 Teams
wc_teams = [
    'Australia', 'Bangladesh', 'India', 'Netherlands',
    'Pakistan', 'South Africa', 'England', 'Ireland',
    'New Zealand', 'Scotland', 'Sri Lanka', 'West Indies'
]

delivery_df = delivery_df[delivery_df['batting_team'].isin(wc_teams)]
delivery_df = delivery_df[delivery_df['bowling_team'].isin(wc_teams)]

print(f'Rows after filter: {len(delivery_df)}')
print(delivery_df['batting_team'].unique())

Rows after filter: 82653
['Australia' 'New Zealand' 'Sri Lanka' 'India' 'England' 'West Indies'
 'Pakistan' 'South Africa' 'Ireland' 'Bangladesh' 'Scotland' 'Netherlands']


In [9]:
output = delivery_df[['match_id', 'batting_team', 'bowling_team', 'ball',
                        'runs', 'player_dismissed', 'city', 'venue']]
pickle.dump(output, open('dataset_level2_female.pkl', 'wb'))
print('Saved dataset_level2_female.pkl')

Saved dataset_level2_female.pkl


In [10]:
output = pickle.load(open('dataset_level2_female.pkl', 'rb'))

# Fix city nulls using venue
x = np.where(output['city'].isnull(),
             output['venue'].str.split().apply(lambda x: x[0]),
             output['city'])
output['city'] = x
output.drop(columns=['venue'], inplace=True)

# Merge total match runs
total_df = output.groupby('match_id').sum()['runs'].reset_index()
output = output.merge(total_df, on='match_id')  # runs_x = per ball, runs_y = match total

# Fix dtypes
output['runs_x'] = pd.to_numeric(output['runs_x'], errors='coerce')
output['runs_y'] = pd.to_numeric(output['runs_y'], errors='coerce')

# Compute features
output['current_score'] = output.groupby('match_id')['runs_x'].cumsum()
output['over']          = output['ball'].apply(lambda x: str(x).split('.')[0])
output['ball_no']       = output['ball'].apply(lambda x: str(x).split('.')[1])
output['balls_bowled']  = (output['over'].astype('int') * 6) + output['ball_no'].astype('int')
output['crr']           = round((output['current_score'] * 6) / output['balls_bowled'], 2)

output['player_dismissed'] = output['player_dismissed'].apply(lambda x: 0 if x == '0' else 1)
output['player_dismissed'] = output['player_dismissed'].astype('int')
output['player_dismissed'] = output.groupby('match_id')['player_dismissed'].cumsum()
output['wickets_left']     = 10 - output['player_dismissed']

print(output.shape)
output.head()

(82653, 14)


,match_id,batting_team,bowling_team,ball,runs_x,player_dismissed,city,runs_y,current_score,over,ball_no,balls_bowled,crr,wickets_left
0,6,Australia,New Zealand,0.1,1,0,Melbourne,151,1,0,1,1,6.0,10
1,6,Australia,New Zealand,0.2,0,0,Melbourne,151,1,0,2,2,3.0,10
2,6,Australia,New Zealand,0.3,0,0,Melbourne,151,1,0,3,3,2.0,10
3,6,Australia,New Zealand,0.4,1,0,Melbourne,151,2,0,4,4,3.0,10
4,6,Australia,New Zealand,0.5,0,0,Melbourne,151,2,0,5,5,2.4,10


In [11]:
final_df = output[['match_id', 'batting_team', 'bowling_team', 'runs_x',
                    'current_score', 'balls_bowled', 'wickets_left', 'crr',
                    'city', 'runs_y']]
final_df = final_df.sample(final_df.shape[0])
final_df['balls_left'] = 120 - final_df['balls_bowled']
final_df['balls_left'] = final_df['balls_left'].apply(lambda x: 0 if x < 0 else x)
final_df['crr']        = round((final_df['current_score'] * 6) / final_df['balls_bowled'], 2)
final_df.drop(columns=['balls_bowled'], inplace=True)

# Last 5 overs rolling (30 balls)
groups    = final_df.groupby('match_id')
match_ids = final_df['match_id'].unique()
last_five = []
for id in match_ids:
    last_five.extend(groups.get_group(id)['runs_x'].rolling(window=30).sum().values.tolist())

final_df['last_five'] = last_five
final_df.dropna(inplace=True)

# Filter eligible cities
eligible_cities = final_df['city'].value_counts()[
    final_df['city'].value_counts() > 100
].index.tolist()
final_df = final_df[final_df['city'].isin(eligible_cities)]

print(f'Eligible cities: {eligible_cities}')
print(f'Rows remaining: {len(final_df)}')
final_df.head()

Eligible cities: ['Sylhet', 'Colombo', 'Dublin', 'Sharjah', 'Birmingham', 'Sydney', 'Cape Town', 'Karachi', 'Kuala Lumpur', 'Melbourne', 'Mumbai', 'Canberra', 'St Lucia', 'Dubai', 'Chelmsford', 'Galle', 'Wellington', 'Guyana', 'East London', 'Benoni', 'Potchefstroom', 'Paarl', 'Dambulla', 'Derby', 'Navi Mumbai', 'Abu Dhabi', 'North Sound', 'Bridgetown', 'Chennai', 'Amstelveen', 'Deventer', 'London', 'Durban', 'Taunton', 'Bristol', 'Dhaka', 'Hamilton', 'Delhi', 'Brisbane', 'Brighton', 'Gqeberha', 'Murcia', 'Mount Maunganui', 'Gros Islet', 'Northampton', 'Dunedin', 'Bready', 'Belfast', 'Adelaide', 'Bangalore', 'Auckland', 'Guanggong', 'Gold Coast', 'Loughborough', 'Christchurch', 'Nelson', 'Doha', 'Antigua', 'Perth', 'Surat', 'Utrecht', 'Lahore', 'Almeria', 'Johannesburg', 'Hangzhou', 'Bangkok', 'Multan', 'Guwahati', 'Queenstown', 'Kingstown', 'Chester-le-Street', 'Kathmandu', 'Basseterre', 'Mirpur', 'Chattogram', 'Kirtipur', 'Lucknow', 'Trinidad', 'Thiruvananthapuram', 'Dundee', 'Canter

,match_id,batting_team,bowling_team,runs_x,current_score,wickets_left,crr,city,runs_y,balls_left,last_five
8458,217,New Zealand,Pakistan,1,69,9,6.57,Guyana,144,57,41.0
56353,1962,England,West Indies,2,116,6,6.89,Chelmsford,144,19,41.0
13656,385,England,Pakistan,1,64,8,7.68,Canberra,158,70,40.0
70815,2902,Pakistan,Ireland,1,129,4,7.04,Dublin,148,10,39.0
59414,2259,Pakistan,South Africa,0,47,7,7.42,Potchefstroom,180,82,39.0


In [12]:
X = final_df.drop(columns=['match_id', 'runs_x', 'runs_y'])
y = final_df['runs_y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'),
     ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

pipe = Pipeline(steps=[
    ('step1', trf),
    ('step2', StandardScaler()),
    ('step3', XGBRegressor(n_estimators=1000, learning_rate=0.2, max_depth=12, random_state=1))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print('R2 Score :', r2_score(y_test, y_pred))
print('MAE      :', mean_absolute_error(y_test, y_pred))

R2 Score : 0.9612330794334412
MAE      : 2.7237850789362765


In [13]:
pickle.dump(pipe, open('pipe_female.pkl', 'wb'))
print('Model saved as pipe_female.pkl ✅')

Model saved as pipe_female.pkl ✅
